# 토큰 수 테스트 노트북

이 노트북은 다양한 파일 형식의 토큰 수를 분석하고 테스트


In [ ]:
# 모듈 import 및 설정
import sys
from pathlib import Path
from typing import Dict, List, cast

from omegaconf import DictConfig, OmegaConf

# 경로 설정 (상대 경로 사용)
project_root: Path = Path("../..").resolve()
source_root: Path = project_root / "src"

# sys.path에 프로젝트 루트 추가
sys.path.insert(0, str(project_root))

# config 로드
config_path: Path = source_root / "conf" / "config.yaml"
cfg: DictConfig = cast(DictConfig, OmegaConf.load(config_path))

# 모듈 import
from src.core.chunker.token_utils import MAX_TOKENS, count_tokens
from src.core.loader_router.loader import get_loader

print("✅ 모듈 import 완료")
print(f"프로젝트 루트: {project_root}")
print(f"Source 루트: {source_root}")
print(f"MAX_TOKENS: {MAX_TOKENS}")


In [ ]:
# 토큰 통계 계산 함수
from typing import Tuple


def calculate_token_statistics(all_files: List[Path]) -> Dict[str, int]:
    """전체 파일들의 확장자별 토큰 수와 총 토큰 수 계산"""
    print("=== 토큰 통계 계산 ===")

    if not all_files:
        print("❌ 계산할 파일이 없습니다!")
        return {}

    total_files: int = len(all_files)
    print(f"처리할 파일 수: {total_files}개")

    ext_token_counts: Dict[str, int] = {}
    ext_file_tokens: Dict[str, List[Tuple[str, int]]] = {}  # 확장자별 (파일명, 토큰수) 리스트
    total_tokens: int = 0
    processed_files: int = 0
    failed_files: int = 0

    for i, file_path in enumerate(all_files, 1):
        try:
            # 진행상황 표시 (5% 단위로)
            if i % max(1, total_files // 20) == 0 or i == total_files:
                progress: float = (i / total_files) * 100
                print(f"진행률: {progress:.1f}% ({i}/{total_files}) - 현재 파일: {file_path.name}")
                print(f"  누적 처리: {processed_files}개 성공, {failed_files}개 실패")
                print(f"  누적 토큰: {total_tokens:,}개")
                print()

            loader = get_loader(str(file_path), cfg)
            if loader:
                from langchain_core.documents import Document

                try:
                    docs: List[Document] = loader.load()
                    if docs:
                        file_tokens: int = sum(count_tokens(doc.page_content) for doc in docs)
                        ext: str = file_path.suffix.lower()
                        ext_token_counts[ext] = ext_token_counts.get(ext, 0) + file_tokens

                        # 확장자별 파일 토큰 수 저장
                        if ext not in ext_file_tokens:
                            ext_file_tokens[ext] = []
                        ext_file_tokens[ext].append((file_path.name, file_tokens))

                        total_tokens += file_tokens
                        processed_files += 1
                    else:
                        # 빈 문서는 실패가 아닌 정상 처리
                        processed_files += 1
                except Exception as e:
                    # 모든 예외 처리 (IndexError 포함)
                    print(f"❌ 파일 처리 실패: {file_path.name} - 오류: {str(e)}")
                    failed_files += 1
            else:
                failed_files += 1
        except Exception as e:
            print(f"❌ 파일 처리 실패: {file_path.name} - 오류: {str(e)}")
            failed_files += 1

    print(f"\n처리 완료: {processed_files}개 파일, 실패: {failed_files}개 파일")

    print("\n=== 확장자별 상세 통계 ===")
    for ext in sorted(ext_token_counts.keys()):
        tokens: int = ext_token_counts[ext]
        file_tokens_list: List[Tuple[str, int]] = ext_file_tokens[ext]

        # 통계 계산
        token_values: List[int] = [tokens for _, tokens in file_tokens_list]
        min_tokens: int = min(token_values)
        max_tokens: int = max(token_values)
        avg_tokens: float = sum(token_values) / len(token_values)
        median_tokens: float = sorted(token_values)[len(token_values) // 2]

        # 최소/최대 토큰 파일 찾기
        min_file: str = next(name for name, t in file_tokens_list if t == min_tokens)
        max_file: str = next(name for name, t in file_tokens_list if t == max_tokens)

        print(f"\n{ext}:")
        print(f"  총 토큰 수: {tokens:,} 토큰")
        print(f"  파일 수: {len(file_tokens_list)}개")
        print(f"  최소 토큰: {min_tokens:,} 토큰 ({min_file})")
        print(f"  최대 토큰: {max_tokens:,} 토큰 ({max_file})")
        print(f"  평균 토큰: {avg_tokens:,.1f} 토큰")
        print(f"  중간값 토큰: {median_tokens:,.1f} 토큰")

    print(f"\n=== 총 토큰 수 ===")
    print(f"총 토큰 수: {total_tokens:,} 토큰")

    print(f"\n=== 예상 API 비용 (배치 API) ===")
    batch_cost_per_1k: float = 0.00001  # 1M tokens = $0.01 → 1K = $0.00001
    estimated_cost: float = (total_tokens / 1000) * batch_cost_per_1k
    print(f"예상 비용: ${estimated_cost:.4f}")
    print(f"1K 토큰당 비용: ${batch_cost_per_1k}")

    return ext_token_counts


In [ ]:
# 스프레드시트 전용 토큰 통계 계산 함수
def calculate_spreadsheet_token_statistics(all_files: List[Path]) -> Dict[str, int]:
    """스프레드시트 파일들의 확장자별 토큰 수와 총 토큰 수 계산"""
    print("=== 스프레드시트 파일 토큰 통계 계산 ===")

    if not all_files:
        print("❌ 계산할 파일이 없습니다!")
        return {}

    total_files: int = len(all_files)
    print(f"처리할 파일 수: {total_files}개")

    ext_token_counts: Dict[str, int] = {}
    ext_file_tokens: Dict[str, List[Tuple[str, int]]] = {}  # 확장자별 (파일명, 토큰수) 리스트
    total_tokens: int = 0
    processed_files: int = 0
    failed_files: int = 0

    for i, file_path in enumerate(all_files, 1):
        try:
            # 진행상황 표시 (10% 단위로)
            if i % max(1, total_files // 10) == 0 or i == total_files:
                progress: float = (i / total_files) * 100
                print(f"진행률: {progress:.1f}% ({i}/{total_files}) - 현재 파일: {file_path.name}")
                print(f"  누적 처리: {processed_files}개 성공, {failed_files}개 실패")
                print(f"  누적 토큰: {total_tokens:,}개")
                print()

            loader = get_loader(str(file_path), cfg)
            if loader:
                from langchain_core.documents import Document

                try:
                    docs: List[Document] = loader.load()
                    if docs:
                        file_tokens: int = sum(count_tokens(doc.page_content) for doc in docs)
                        ext: str = file_path.suffix.lower()
                        ext_token_counts[ext] = ext_token_counts.get(ext, 0) + file_tokens

                        # 확장자별 파일 토큰 수 저장
                        if ext not in ext_file_tokens:
                            ext_file_tokens[ext] = []
                        ext_file_tokens[ext].append((file_path.name, file_tokens))

                        total_tokens += file_tokens
                        processed_files += 1
                    else:
                        # 빈 문서는 실패가 아닌 정상 처리
                        processed_files += 1
                except Exception as e:
                    # 모든 예외 처리 (IndexError 포함)
                    print(f"❌ 파일 처리 실패: {file_path.name} - 오류: {str(e)}")
                    failed_files += 1
            else:
                failed_files += 1
        except Exception as e:
            print(f"❌ 파일 처리 실패: {file_path.name} - 오류: {str(e)}")
            failed_files += 1

    print(f"\n처리 완료: {processed_files}개 파일, 실패: {failed_files}개 파일")

    print("\n=== 확장자별 상세 통계 ===")
    for ext in sorted(ext_token_counts.keys()):
        tokens: int = ext_token_counts[ext]
        file_tokens_list: List[Tuple[str, int]] = ext_file_tokens[ext]

        # 통계 계산
        token_values: List[int] = [tokens for _, tokens in file_tokens_list]
        min_tokens: int = min(token_values)
        max_tokens: int = max(token_values)
        avg_tokens: float = sum(token_values) / len(token_values)
        median_tokens: float = sorted(token_values)[len(token_values) // 2]

        # 최소/최대 토큰 파일 찾기
        min_file: str = next(name for name, t in file_tokens_list if t == min_tokens)
        max_file: str = next(name for name, t in file_tokens_list if t == max_tokens)

        print(f"\n{ext}:")
        print(f"  총 토큰 수: {tokens:,} 토큰")
        print(f"  파일 수: {len(file_tokens_list)}개")
        print(f"  최소 토큰: {min_tokens:,} 토큰 ({min_file})")
        print(f"  최대 토큰: {max_tokens:,} 토큰 ({max_file})")
        print(f"  평균 토큰: {avg_tokens:,.1f} 토큰")
        print(f"  중간값 토큰: {median_tokens:,.1f} 토큰")

    print(f"\n=== 총 토큰 수 ===")
    print(f"총 토큰 수: {total_tokens:,} 토큰")

    print(f"\n=== 예상 API 비용 (배치 API) ===")
    batch_cost_per_1k: float = 0.00001  # 1M tokens = $0.01 → 1K = $0.00001
    estimated_cost: float = (total_tokens / 1000) * batch_cost_per_1k
    print(f"예상 비용: ${estimated_cost:.4f}")
    print(f"1K 토큰당 비용: ${batch_cost_per_1k}")

    return ext_token_counts


In [ ]:
# 개별 파일 분석 함수
from langchain_core.documents import Document


def analyze_file_content(file_path: Path) -> None:
    """특정 파일의 내용과 토큰 수를 자세히 분석"""
    print(f"=== {file_path.name} 파일 내용 분석 ===")

    if not file_path.exists():
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
        return

    print(f"✅ 파일 발견: {file_path}")

    try:
        loader = get_loader(str(file_path), cfg)
        docs: List[Document] = loader.load()

        print(f"문서 수: {len(docs)}개")

        if docs:
            # 모든 문서의 토큰 수 계산
            total_tokens: int = 0
            doc: Document
            for i, doc in enumerate(docs):
                token_count: int = count_tokens(doc.page_content)
                total_tokens += token_count
                print(f"\n--- 문서 {i + 1} ---")
                print(f"메타데이터: {doc.metadata}")

                # row_data가 있는 경우 별도로 출력
                if "row_data" in doc.metadata:
                    print(f"row_data: {doc.metadata['row_data']}")

                print(f"내용 길이: {len(doc.page_content):,} 문자")
                print(f"토큰 수: {token_count:,}")
                print(f"전체 page_content:")
                print(repr(doc.page_content))
                print("-" * 80)

            print(f"\n=== 전체 통계 ===")
            print(f"총 문서 수: {len(docs)}개")
            print(f"총 토큰 수: {total_tokens:,}")
            print(f"평균 토큰 수: {total_tokens / len(docs):,.1f}")

        else:
            print("❌ 문서가 로드되지 않았습니다.")

    except Exception as e:
        print(f"❌ 파일 처리 실패: {e}")


In [ ]:
# 테스트 파일 찾기
test_files_dir: Path = project_root / "notebooks" / "test-files"

# 모든 테스트 파일 찾기
all_files: List[Path] = []
supported_extensions: List[str] = [
    ".txt",
    ".md",
    ".pdf",
    ".docx",
    ".doc",
    ".hwp",
    ".org",
    ".rtf",
    ".xlsx",
    ".xls",
    ".csv",
    ".tsv",
    ".pptx",
    ".ppt",
]

ext: str
for ext in supported_extensions:
    ext_files: List[Path] = list(test_files_dir.rglob(f"*{ext}"))
    all_files.extend(ext_files)

print(f"총 테스트 파일 수: {len(all_files)}개")

# 스프레드시트 파일만 찾기
spreadsheet_files: List[Path] = []
spreadsheet_extensions: List[str] = [".xlsx", ".xls", ".csv", ".tsv"]
for ext in spreadsheet_extensions:
    spreadsheet_ext_files: List[Path] = list(test_files_dir.rglob(f"*{ext}"))
    spreadsheet_files.extend(spreadsheet_ext_files)

print(f"스프레드시트 파일 수: {len(spreadsheet_files)}개")

# 확장자별 파일 수 확인
ext_counts: Dict[str, int] = {}
file_path: Path
for file_path in all_files:
    ext = file_path.suffix.lower()
    ext_counts[ext] = ext_counts.get(ext, 0) + 1

print(f"\n확장자별 파일 수:")
for ext in sorted(ext_counts.keys()):
    print(f"  {ext}: {ext_counts[ext]}개")


In [ ]:
print("=== 전체 파일 토큰 통계 계산 ===")
all_token_stats: Dict[str, int] = calculate_token_statistics(all_files)


In [ ]:
# poc-shared-strings.xlsx 파일 분석
# xlsx 최대 토큰
# 4,403,297,250 토큰
poc_file: Path = test_files_dir / "spreadsheets" / "xls" / "poc-shared-strings.xlsx"
analyze_file_content(poc_file)


In [ ]:
# 46904.xls 파일 분석
# xls 최대 토큰
# 492,374 토큰
xls_file: Path = test_files_dir / "spreadsheets" / "xls" / "46904.xls"
analyze_file_content(xls_file)


In [ ]:
# ratings.csv 파일 분석
# csv 최대 토큰
# 16,933,108 토큰
csv_file: Path = test_files_dir / "spreadsheets" / "csv" / "ratings.csv"
analyze_file_content(csv_file)


In [ ]:
# SquareMacro.xls 파일 분석
# xls 최소 토큰
# 0 토큰
xls_min_file: Path = test_files_dir / "spreadsheets" / "xls" / "SquareMacro.xls"
analyze_file_content(xls_min_file)

In [ ]:
# 스프레드시트 파일 토큰 통계 계산
print("=== 스프레드시트 파일 토큰 통계 계산 ===")
spreadsheet_stats: Dict[str, int] = calculate_spreadsheet_token_statistics(spreadsheet_files)


In [ ]:
# row_data 생성 로직 확인
import sys
from pathlib import Path
from typing import List

sys.path.append("src")
from app.config import load_config
from core.loader_router.enhanced_router import load_document_with_router

cfg: DictConfig = load_config()
root: Path = Path("/Users/wonsik/Desktop/Voyager/voyager-app-backend/notebooks/test-files")
targets: List[Path] = [
    p for p in root.rglob("*") if p.suffix.lower() in {".csv", ".tsv", ".xlsx", ".xls"}
]


def debug_row_data_creation() -> None:
    p: Path
    for p in targets[:3]:  # 처음 3개만 테스트
        try:
            docs: List[Document] = load_document_with_router(p, cfg)
            print(f"\n=== {p.name} ===")
            print(f"문서 수: {len(docs)}")

            if docs:
                # 첫 번째 문서의 row_data 생성 과정 시뮬레이션
                doc: Document = docs[0]

                # 안전한 메타데이터 접근
                try:
                    if hasattr(doc, "metadata"):
                        metadata_dict = getattr(doc, "metadata", {})
                        if "row_data" in metadata_dict:
                            row_data = metadata_dict["row_data"]
                            print(f"row_data 타입: {type(row_data)}")
                            print(f"row_data 내용: {row_data}")

                            # 실제 process_row 로직 시뮬레이션
                            if isinstance(row_data, dict):
                                meaningful_content: List[str] = []

                                # 딕셔너리 항목 처리
                                for key, value in row_data.items():
                                    key_str: str = str(key)
                                    if key_str != "text" and value is not None:
                                        value_str: str = str(value)
                                        if value_str.strip():
                                            meaningful_content.append(value_str)

                                print(f"의미있는 컨텐츠: {meaningful_content}")
                                print(f"의미있는 컨텐츠 수: {len(meaningful_content)}")

                                if meaningful_content:
                                    first_content: str = meaningful_content[0]
                                    print(f"첫 번째 의미있는 컨텐츠: {first_content}")
                                    print(f"타입: {type(first_content)}")
                                    print(f"strip() 가능 여부: {hasattr(first_content, 'strip')}")
                except (AttributeError, TypeError, KeyError) as meta_error:
                    print(f"메타데이터 접근 오류: {meta_error}")

        except Exception as e:
            print(f"❌ {p.name}: {e}")
            import traceback

            traceback.print_exc()


debug_row_data_creation()


In [ ]:
# 문서 수 0인 파일들에서 실제 오류 발생 확인
import sys
from pathlib import Path
from typing import List

sys.path.append("src")
from app.config import load_config
from core.loader_router.enhanced_router import load_document_with_router

cfg: DictConfig = load_config()
root: Path = Path(
    "/Users/wonsik/Desktop/Voyager/voyager-app-backend/notebooks/test-files/spreadsheets/xls"
)

# 문서 수 0인 파일들만 테스트
zero_doc_files: List[str] = [
    "53282b.xlsx",
    "link-external-workbook-b.xlsx",
    "50096.xlsx",
    "NewStyleConditionalFormattings.xls",
    "unicodeSheetName.xlsx",
    "58760.xlsx",
]


def test_zero_doc_files() -> None:
    filename: str
    for filename in zero_doc_files:
        file_path: Path = root / filename
        if file_path.exists():
            try:
                docs: List[Document] = load_document_with_router(file_path, cfg)
                print(f"✅ {filename}: 문서 수 {len(docs)}")

                # process_row 시뮬레이션 - 빈 docs에서 접근 시도
                if len(docs) == 0:
                    print(f"  ⚠️  빈 문서 리스트 - process_row에서 오류 가능성")
                    # 실제 process_row에서 docs[0]에 접근하려고 하면 IndexError 발생
                    try:
                        doc: Document = docs[0]  # 이 부분에서 IndexError 발생
                        print(f"  문서 발견: {doc}")  # 실제로는 실행되지 않음
                    except IndexError as e:
                        print(f"  ❌ IndexError 발생: {e}")

            except Exception as e:
                print(f"❌ {filename}: {e}")


test_zero_doc_files()
